EEG Condition Classification Using Machine Learning

This notebook extends the previous EEG analysis by treating the two subject conditions (eyes-open and eyes-closed)  as a binary classification problem.

The first notebook investigated the Berger effect using spectral analysis, successfully finding it to be present in the subject's dataset. This second experiment builds on that work by exploring whether machine learning can recognise the differences between eyes-open and eyes-closed EEG recordings.

This experiment is exploratory and intended as a self-introduction to ML in an EEG context.

In [3]:
from pathlib import Path
import sys

print("Current working directory:", Path.cwd())
print("sys.path:")
for p in sys.path:
    print(" ", p)

Current working directory: c:\Users\aryan\OneDrive\Documents\EEG visualisation project\EEG analysis pipeline
sys.path:
  C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\python313.zip
  C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\DLLs
  C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib
  C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0
  
  C:\Users\aryan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages
  C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\site-packages


In [ ]:
import numpy as np
import mne

from eeg_pipeline import load_subject, preprocess_raw, extract_events

raw, closed_trigger, open_trigger = load_subject("subject_01")

assert isinstance(raw, mne.io.RawArray)
assert len(raw.ch_names) == 16
assert raw.n_times == 119808
assert raw.info["sfreq"] == 512

assert closed_trigger.shape == (119808,)
assert open_trigger.shape == (119808,)

print("load_subject passed")
print(raw)

Creating RawArray with float64 data, n_channels=16, n_times=119808
    Range : 0 ... 119807 =      0.000 ...   233.998 secs
Ready.
load_subject passed
<RawArray | 16 x 119808 (234.0 s), ~14.6 MiB, data loaded>


In [5]:

raw_clean = preprocess_raw(raw)

assert isinstance(raw_clean, mne.io.RawArray)
assert raw_clean is not raw
assert raw_clean.n_times == raw.n_times
assert len(raw_clean.ch_names) == len(raw.ch_names)

print("preprocess_raw passed")
print("Original:", raw.info["highpass"], raw.info["lowpass"])
print("Filtered:", raw_clean.info["highpass"], raw_clean.info["lowpass"])

preprocess_raw passed
Original: 0.0 256.0
Filtered: 1.0 40.0


In [6]:
from eeg_pipeline import extract_events

events = extract_events(closed_trigger, open_trigger)

assert isinstance(events, np.ndarray)
assert events.shape == (10, 3)

assert np.sum(events[:, 2] == 1) == 5
assert np.sum(events[:, 2] == 2) == 5

assert np.array_equal(np.unique(events[:, 2]), np.array([1, 2]))
assert np.all(np.diff(events[:, 0]) > 0)

print("extract_events passed")
print(events)

extract_events passed
[[ 35511      0      1]
 [ 45179      0      2]
 [ 53413      0      1]
 [ 61886      0      2]
 [ 69812      0      1]
 [ 78140      0      2]
 [ 85802      0      1]
 [ 93556      0      2]
 [101416      0      1]
 [109193      0      2]]


In [6]:
import numpy as np
import mne

from eeg_pipeline import (
    load_subject,
    inspect_and_mark_bad_channels,
    preprocess_raw,
    extract_events,
    make_epochs,
    compute_alpha_power,
)

# 1. Load the subject
raw, closed_trigger, open_trigger = load_subject("subject_01")

assert isinstance(raw, mne.io.RawArray)
assert len(raw.ch_names) == 16
assert raw.n_times == 119808
assert raw.info["sfreq"] == 512
assert closed_trigger.shape == (119808,)
assert open_trigger.shape == (119808,)

print("1. load_subject passed")
print(raw)

# 2. Inspect and optionally mark bad channels
# Leave bad_channels as None if you do not want to mark any yet.
raw_checked = inspect_and_mark_bad_channels(
    raw,
    bad_channels=None,
    show_plot=False,
)

assert isinstance(raw_checked, mne.io.RawArray)
assert raw_checked is not raw

print("2. inspect_and_mark_bad_channels passed")
print("Bad channels:", raw_checked.info["bads"])

# 3. Preprocess / filter
raw_clean = preprocess_raw(raw_checked)

assert isinstance(raw_clean, mne.io.RawArray)
assert raw_clean is not raw_checked
assert raw_clean.n_times == raw_checked.n_times
assert not np.array_equal(
    raw_checked.get_data(),
    raw_clean.get_data(),
)

print("3. preprocess_raw passed")

# 4. Extract events
events = extract_events(closed_trigger, open_trigger)

assert isinstance(events, np.ndarray)
assert events.shape == (10, 3)
assert np.sum(events[:, 2] == 1) == 5
assert np.sum(events[:, 2] == 2) == 5
assert np.all(np.diff(events[:, 0]) > 0)

print("4. extract_events passed")
print(events)

# 5. Create epochs
event_id = {
    "eyes_closed": 1,
    "eyes_open": 2,
}

epochs = make_epochs(
    raw_clean,
    events,
    event_id,
    epoch_length=10.0,
)

assert isinstance(epochs, mne.Epochs)
assert len(epochs) == 10
assert len(epochs["eyes_closed"]) == 5
assert len(epochs["eyes_open"]) == 5
assert epochs.get_data().shape == (10, 16, 5121)

print("5. make_epochs passed")
print(epochs)

# 6. Compute alpha power
alpha_power_all = compute_alpha_power(epochs)

assert isinstance(alpha_power_all, dict)
assert set(alpha_power_all.keys()) == {"eyes_closed", "eyes_open"}
assert all(value > 0 for value in alpha_power_all.values())

print("6. compute_alpha_power passed")
print("Alpha power, all EEG channels:")
print(alpha_power_all)

# 7. Compute alpha power at posterior channels
posterior_channels = ["P7", "P3", "PZ", "P4", "P8", "O1", "OZ", "O2"]

alpha_power_posterior = compute_alpha_power(
    epochs,
    picks=posterior_channels,
)

assert set(alpha_power_posterior.keys()) == {"eyes_closed", "eyes_open"}
assert all(value > 0 for value in alpha_power_posterior.values())

print("7. posterior alpha power passed")
print("Alpha power, posterior channels:")
print(alpha_power_posterior)

print("\nAll functions passed.")

Creating RawArray with float64 data, n_channels=16, n_times=119808
    Range : 0 ... 119807 =      0.000 ...   233.998 secs
Ready.
1. load_subject passed
<RawArray | 16 x 119808 (234.0 s), ~14.6 MiB, data loaded>
Bad channels: []
2. inspect_and_mark_bad_channels passed
Bad channels: []
3. preprocess_raw passed
4. extract_events passed
[[ 35511      0      1]
 [ 45179      0      2]
 [ 53413      0      1]
 [ 61886      0      2]
 [ 69812      0      1]
 [ 78140      0      2]
 [ 85802      0      1]
 [ 93556      0      2]
 [101416      0      1]
 [109193      0      2]]
5. make_epochs passed
<Epochs | 10 events (all good), 0 – 10 s (baseline off), ~6.3 MiB, data loaded,
 'eyes_closed': 5
 'eyes_open': 5>
6. compute_alpha_power passed
Alpha power, all EEG channels:
{'eyes_closed': 8.184139126724855, 'eyes_open': 4.875976724821354}
7. posterior alpha power passed
Alpha power, posterior channels:
{'eyes_closed': 13.41819717104067, 'eyes_open': 7.1178821797970295}

All functions passed.


In [10]:
print("Posterior alpha power")
print(f"Eyes closed: {alpha_power_posterior['eyes_closed']:.3e}")
print(f"Eyes open:   {alpha_power_posterior['eyes_open']:.3e}")

print("\nAll-channel alpha power")
print(f"Eyes closed: {alpha_power_all['eyes_closed']:.3e}")
print(f"Eyes open:   {alpha_power_all['eyes_open']:.3e}")

Posterior alpha power
Eyes closed: 1.342e+01
Eyes open:   7.118e+00

All-channel alpha power
Eyes closed: 8.184e+00
Eyes open:   4.876e+00
